In [ ]:
!pip install -q optuna

In [ ]:
!pip install -q pymorphy3

# Импорты

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
import requests
import pymorphy3

In [ ]:
import joblib
from datetime import datetime

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.multiclass import OneVsRestClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split, cross_validate, KFold

from sklearn.metrics import classification_report, hamming_loss, f1_score

In [ ]:
url = "https://raw.githubusercontent.com/stopwords-iso/stopwords-ru/refs/heads/master/stopwords-ru.txt"
req = requests.get(url)
stop_words = req.text.split("\n")

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_excel("/content/Комментарии размечено_2.xlsx")
df.head(3)

# Предобработка

In [ ]:
target_columns = ['Зарплата', 'Заработная плата']
df['Зарплата'] = df[target_columns].max(axis=1)

df = df.drop(columns=['Заработная плата'])

In [ ]:
target_columns = ['Социальный пакет', 'Соц. Пакет']
df['Социальный пакет'] = df[target_columns].max(axis=1)

df = df.drop(columns=['Соц. Пакет'])
df.info()

In [ ]:
 #df.to_excel('markup_results.xlsx', index=False)

In [ ]:
df = df.drop(columns=['№', 'Столбец2']) #техничекие поля
df = df.drop(columns=['Сотрудник или студент', 'Есть почтовый адрес', 'Информированность и участие в решениях']) #отсутствует в датасете

In [ ]:
df.info()

In [ ]:
df = df.drop_duplicates()

df = df.reset_index(drop=True)
df.info()

In [ ]:
df['Длина комментария'] = df['Комментарий'].str.split().str.len()

In [ ]:
feature_columns = [col for col in df.columns if col not in ['id', 'Комментарий']]

df = df.copy()
df[feature_columns] = df[feature_columns].fillna(0).astype(int)

In [ ]:
df.insert(0, 'id', range(len(df)))

# ИИ

## Создание json для разметки

In [ ]:

feature_columns = [col for col in df.columns if col not in ['id', 'comment']]

df_filled = df.copy()

df_filled[feature_columns] = df_filled[feature_columns].fillna(0).astype(int)

json_data = []
for _, row in df_filled.iterrows():
    item = {
        "id": int(row['id']),
        "text": row['comment'],
        "labels": row[feature_columns].to_dict()
    }
    json_data.append(item)

# Сохранение в файл
with open('Комментарии_размечено_clean_for_agent_withid.json', 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)

In [ ]:

df_unique = df.dropna(subset=['comment']).drop_duplicates(subset=['comment'])

unlabeled_data = []

for _, row in df_unique.iterrows():
    unlabeled_data.append({
        "id": int(row['id']),
        "text": str(row['comment']),
        "status": "unlabeled"
    })

with open('Комментарии_to_label_withid_all.json', 'w', encoding='utf-8') as f:
    json.dump(unlabeled_data, f, ensure_ascii=False, indent=4)

### Оценка

In [ ]:
with open('/content/labeled_100_1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_results = pd.json_normalize(data)
df_results.columns = [col.replace('labels.', '') for col in df_results.columns]

df_check = pd.merge(
    df[['id', 'Комментарий']],
    df_results,
    on='id',
    how='right'
)

df_check.head()

In [ ]:
df_check.info()

In [ ]:
target_columns = ['опасения в анонимности', 'опасения in анонимности']
df_check['опасения в анонимности'] = df_check[target_columns].max(axis=1)
df_check = df_check.drop(columns=['опасения in анонимности'])

In [ ]:
target_columns = ['Взаимодействие с пациентами', 'Взаимодействие with пациентами']
df_check['Взаимодействие с пациентами'] = df_check[target_columns].max(axis=1)
df_check = df_check.drop(columns=['Взаимодействие with пациентами'])

In [ ]:
df.columns

In [ ]:
comparison_df.info()

In [ ]:

target_columns = ['Зарплата', 'Дефицит',
       'Коллеги', 'Востребованность работы', 'Содержание работы',
       'Руководство и репутация организации', 'Инфраструктура региона',
       'Возможности для развития', 'Профессиональные риски',
       'Стабильность и защищенность', 'Рабочие процессы',
       'Оснащенность организации', 'Условия работы', 'Здравоохранение региона',
       'График работы', 'Взаимодействие с пациентами', 'Социальный пакет',
       'Нет комментариев', 'Мусор', 'Комментарии к опроснику',
       'Благодарность за опросник', 'Скептицизм к изменениям',
       'Надежда на изменения', 'Позитив-неуточненно', 'опасения в анонимности',
       'Негатив-неуточненно', 'Физическое здоровье', 'Общественное здоровье',
       'Заставили']


comparison_df = pd.merge(
    df[['id'] + target_columns],
    df_check[['id'] + target_columns],
    on='id',
    suffixes=('_true', '_pred')
)

y_test_num = comparison_df[[col + '_true' for col in target_columns]].fillna(0).astype(int)
y_pred_num = comparison_df[[col + '_pred' for col in target_columns]].fillna(0).astype(int)

# 4. Вывод отчета
print("Отчет о качестве разметки ИИ-агента:")
print(classification_report(
    y_test_num,
    y_pred_num,
    target_names=target_columns,
    zero_division=0
))

### Codex

In [ ]:
with open('/content/Комментарии_labeled_all.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_codex = pd.json_normalize(data)
df_codex.columns = [col.replace('labels.', '') for col in df_codex.columns]

df_codex_check= pd.merge(
    df[['id', 'Комментарий']],
    df_codex,
    on='id',
    how='right'
)

In [ ]:
df_codex_check.info()

In [ ]:

target_columns = ['Зарплата', 'Дефицит',
       'Коллеги', 'Востребованность работы', 'Содержание работы',
       'Руководство и репутация организации', 'Инфраструктура региона',
       'Возможности для развития', 'Профессиональные риски',
       'Стабильность и защищенность', 'Рабочие процессы',
       'Оснащенность организации', 'Условия работы', 'Здравоохранение региона',
       'График работы', 'Взаимодействие с пациентами', 'Социальный пакет',
       'Нет комментариев', 'Мусор', 'Комментарии к опроснику',
       'Благодарность за опросник', 'Скептицизм к изменениям',
       'Надежда на изменения', 'Позитив-неуточненно', 'опасения в анонимности',
       'Негатив-неуточненно', 'Физическое здоровье', 'Общественное здоровье',
       'Заставили']

comparison_df = pd.merge(
    df[['id'] + target_columns],
    df_codex_check[['id'] + target_columns],
    on='id',
    suffixes=('_true', '_pred')
)

y_test_num = comparison_df[[col + '_true' for col in target_columns]].fillna(0).astype(int)
y_pred_num = comparison_df[[col + '_pred' for col in target_columns]].fillna(0).astype(int)

print("Отчет о качестве разметки Codex:")
print(classification_report(
    y_test_num,
    y_pred_num,
    target_names=target_columns,
    zero_division=0
))

# Предобработка текста

In [ ]:
morph = pymorphy3.MorphAnalyzer()
cache = {}

def clean_text_full(text):
    if not isinstance(text, str) or not text.strip():
        return ""

    text = re.sub(r'[^а-яА-ЯёЁ ]', ' ', text.lower())
    words = text.split()

    result = []
    for word in words:
        # Лемматизируем всё, что длиннее 1 буквы
        if len(word) > 1 or word == 'я':
            if word not in cache:
                cache[word] = morph.parse(word)[0].normal_form
            result.append(cache[word])

    return " ".join(result)

df['comment'] = df['Комментарий'].apply(clean_text_full)
df['Комментарий'] = df['Комментарий'].fillna('').astype(str)
df['comment'] = df['comment'].fillna('').astype(str)

In [ ]:
df['comment'] = df['comment'].astype(str)
df['Комментарий'] = df['Комментарий'].astype(str)

In [ ]:
df[['comment', 'Комментарий']]

In [ ]:
df = df.drop(columns='Комментарий')

In [ ]:
y_columns = [c for c in df.columns if c not in (['Комментарий', 'comment_full'])]
df = df.fillna(0) #тк стоят 1 либо пустота

In [ ]:
df_input = df.copy()

# ML

In [ ]:
def predict(df_given):
  X = df_given[['comment', 'Длина комментария']]
  y = df_given.drop(['comment', 'Длина комментария','id'], axis=1)

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  preprocessor = ColumnTransformer(
      transformers=[
          ('text', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'comment'),
          ('num', StandardScaler(), ['Длина комментария'])
      ]
  )

  pipeline = Pipeline([
      ('preprocessor', preprocessor),
      ('clf', MultiOutputClassifier(LogisticRegression(solver='liblinear', class_weight='balanced')))
  ])

  pipeline.fit(X_train, y_train)

  y_pred = pipeline.predict(X_test)

  y_test_num = y_test.values.astype(int)
  y_pred_num = y_pred.astype(int)

  print(classification_report(y_test_num, y_pred_num, target_names=y.columns, zero_division=0))

In [ ]:
def predict_1000features(df_given):
  X = df_given[['comment', 'Длина комментария']]
  y = df_given.drop(['comment', 'Длина комментария', 'id'], axis=1)

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  preprocessor = ColumnTransformer(
      transformers=[
          ('text', TfidfVectorizer(max_features=1000, ngram_range=(1, 2)), 'comment'),
          ('num', StandardScaler(), ['Длина комментария'])
      ]
  )

  pipeline = Pipeline([
      ('preprocessor', preprocessor),
      ('clf', MultiOutputClassifier(LogisticRegression(solver='liblinear', class_weight='balanced')))
  ])

  pipeline.fit(X_train, y_train)

  y_pred = pipeline.predict(X_test)

  y_test_num = y_test.values.astype(int)
  y_pred_num = y_pred.astype(int)

  print(classification_report(y_test_num, y_pred_num, target_names=y.columns, zero_division=0))

In [ ]:
predict_1000features(df)

In [ ]:
predict(df)

# Удаление категорий

## На основе кореляций

In [ ]:
def draw_corr(df_given):
  labels_df = df_given.drop(columns=['comment', 'Длина комментария'], errors='ignore')

  # Пирсона, для 0/1 это эквивалентно коэффициенту Мэтьюса
  corr = labels_df.corr()


  plt.figure(figsize=(25, 15))
  # tk таблица симметрична, низ дублирует верх
  mask = np.triu(np.ones_like(corr, dtype=bool))

  #
  sns.heatmap(
      corr,
      mask=mask,
      annot=True,
      fmt=".2f",
      cmap='coolwarm',
      center=0,
      square=True,
      cbar_kws={"shrink": .8}
  )

  plt.title('Корреляция между категориями (Labels)', fontsize=15)
  plt.xticks(rotation=45, ha='right')
  plt.tight_layout()
  plt.show()

draw_corr(df)

### Условия работы + оснащенность организации

In [ ]:
target_columns = ['Оснащенность организации', 'Условия работы']
df['Условия работы'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Оснащенность организации'])

In [ ]:
predict(df)

In [ ]:
predict_1000features(df)

### Руководство и репутация + коллеги

In [ ]:
target_columns = ['Руководство и репутация организации', 'Коллеги']
df['Руководство, коллеги и репутация организации'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Руководство и репутация организации', 'Коллеги'])

In [ ]:
predict(df)

### Взаимодействие с пациентами + проф риски + востребованность

In [ ]:
target_columns = ['Взаимодействие с пациентами', 'Профессиональные риски', 'Востребованность работы']
df['Взаимодействие с пациентами, проф риски, востребованность'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Взаимодействие с пациентами', 'Профессиональные риски', 'Востребованность работы'])

In [ ]:
predict(df)

In [ ]:
predict_1000features(df)

## Негатив неуточненный

In [ ]:

def refine_labels_by_phrases(df, phrases, target_col, fallback_col='Негатив-неуточненно'):
    pattern = '|'.join(phrases).replace(' ', r'\s+')

    new_matches = df['comment'].str.contains(pattern, case=False, na=False).astype(int)

    existing_column = df[target_col].fillna(0).astype(int)
    df[target_col] = (existing_column | new_matches)
    df.loc[df[target_col] == 1, fallback_col] = 0

    print(f"Обновлено меток в '{target_col}': {new_matches.sum()}")
    print(f"Осталось в '{fallback_col}': {df[fallback_col].sum()}")

    return df

In [ ]:
exclude_cols = ['comment', 'Длина комментария', 'Негатив-неуточненно']
specific_labels = [col for col in df.columns if col not in exclude_cols]
has_specific_label = df[specific_labels].any(axis=1)

df.loc[has_specific_label, 'Негатив-неуточненно'] = 0
print(f"Осталось записей в 'Негатив-неуточненно': {df['Негатив-неуточненно'].sum()}")

In [ ]:
phrases = [
    'зарплат',
    'оклад'
]

df = refine_labels_by_phrases(df, phrases, 'Зарплата', 'Негатив-неуточненно')

In [ ]:
phrases = [
    'нет слово',
    'воздержаться',
    'бещ комментарий',
    'я всё изложить предоставить я вопрос',
    'я всё равно'
]

df = refine_labels_by_phrases(df, phrases, 'Нет комментариев', 'Негатив-неуточненно')
df[(df['Негатив-неуточненно'] == 1)]['comment']

In [ ]:
phrases = [
    'кошмар',
    'сложно',
    'ужас',
    'комментарий на мой взгяд быть лишний',
    'отрицательный'
]

df = refine_labels_by_phrases(df, phrases, 'Комментарии к опроснику', 'Негатив-неуточненно')
df[(df['Негатив-неуточненно'] == 1)]['comment']

In [ ]:
phrases = [
    'ургентный'
]
df = refine_labels_by_phrases(df, phrases, 'График работы', 'Негатив-неуточненно')

In [ ]:
phrases = [
    'новшество',
    'уничтожить'
]
df = refine_labels_by_phrases(df, phrases, 'Скептицизм к изменениям', 'Негатив-неуточненно')

In [ ]:
phrases = [
    'уйти'
]

df = refine_labels_by_phrases(df, phrases, 'Содержание работы', 'Негатив-неуточненно')

In [ ]:
df = df.drop(columns=['Негатив-неуточненно'])

In [ ]:
predict(df)

In [ ]:
predict_1000features(df)

## Другие маленькие комментарии

In [ ]:
target_columns = ['Стабильность и защищенность', 'Физическое здоровье']
df['Стабильность и защищенность'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Физическое здоровье'])



In [ ]:
target_columns = ['Стабильность и защищенность', 'Взаимодействие с пациентами, проф риски, востребованность']
df['Взаимодействие с пациентами, проф риски, востребованность'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Стабильность и защищенность'])


In [ ]:
target_columns = ['Взаимодействие с пациентами, проф риски, востребованность', 'Содержание работы']
df['Взаимодействие с пациентами, проф риски, востребованность'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Содержание работы'])


In [ ]:
target_columns = ['Нет комментариев', 'Мусор']
df['Нет комментариев'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Мусор'])


In [ ]:
target_columns = ['Заставили', 'опасения в анонимности', 'Комментарии к опроснику']
df['Комментарии к опроснику'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Заставили', 'опасения в анонимности'])


In [ ]:
target_columns = ['Инфраструктура региона', 'Здравоохранение региона']
df['Инфраструктура и здравоохранение региона'] = df[target_columns].max(axis=1)
df = df.drop(columns=['Инфраструктура региона', 'Здравоохранение региона'])


In [ ]:
df.sum()

### Общественное здоровье

In [ ]:
print(df[df['Общественное здоровье'] == 1].comment)

In [ ]:
phrases = [
    'убивать большой количество ненужный отчёт',
    'необходимо выстроить абсолютно новый система',
    'необходимо большой внедрять профилактика'
]

df = refine_labels_by_phrases(df, phrases, 'Инфраструктура и здравоохранение региона', 'Общественное здоровье')
df[(df['Общественное здоровье'] == 1)]['comment']

In [ ]:
df = df.drop(columns=['Общественное здоровье'])

In [ ]:
predict(df)

In [ ]:
draw_corr(df)

# Объединение графика и условий работы

In [ ]:
target_columns = ['График работы', 'Условия работы', 'Рабочие процессы']
df['Условия работы'] = df[target_columns].max(axis=1)
df = df.drop(columns=['График работы', 'Рабочие процессы'])


In [ ]:
predict(df)

# Выбор лучшей модели

## MultiOutputClassifier

In [ ]:
def predict_ext(df_given, classifier_type='logreg'):
    X = df_given[['comment', 'Длина комментария']]
    y = df_given.drop(['comment', 'Длина комментария', 'id'], axis=1).fillna(0).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


    if classifier_type == 'logreg':
        clf = LogisticRegression(solver='liblinear', class_weight='balanced')
    elif classifier_type == 'svc':
        clf = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)
    elif classifier_type == 'rf':
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1)
    elif classifier_type == 'xgb':
        clf = XGBClassifier(eval_metric='logloss', use_label_encoder=False)
    else:
        raise ValueError("Unknown classifier type")

    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'comment'),
            ('num', StandardScaler(), ['Длина комментария'])
        ]
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', MultiOutputClassifier(clf))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"--- Результаты модели: {classifier_type.upper()} ---")
    print(classification_report(y_test, y_pred, target_names=y.columns, zero_division=0))

    return pipeline


In [ ]:
model_res = predict_ext(df, classifier_type='logreg')

In [ ]:
model_res = predict_ext(df, classifier_type='svc')

In [ ]:
model_res =  predict_ext(df, classifier_type='rf')

In [ ]:
model_res = predict_ext(df, classifier_type='xgb')

### 5-fold

In [ ]:
def predict_ext_with_cv(df_given, classifier_type='logreg'):
    X = df_given[['comment', 'Длина комментария']]
    y = df_given.drop(['comment', 'Длина комментария', 'id'], axis=1, errors='ignore').fillna(0).astype(int)

    if classifier_type == 'logreg':
        clf = LogisticRegression(solver='liblinear', class_weight='balanced')
    elif classifier_type == 'svc':
        clf = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)
    elif classifier_type == 'rf':
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1)
    elif classifier_type == 'xgb':
        clf = XGBClassifier(eval_metric='logloss')
    else:
        raise ValueError("Unknown classifier type")

    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'comment'),
            ('num', StandardScaler(), ['Длина комментария'])
        ]
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', MultiOutputClassifier(clf))
    ])

    #5-фолд кросс-валидация
    print(f"5-fold CV для MultiOutput ({classifier_type.upper()})...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    scoring = ['f1_micro', 'precision_micro', 'recall_micro']
    cv_results = cross_validate(pipeline, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    metrics_df = pd.DataFrame({
        'Фолд': [f'Fold {i+1}' for i in range(5)],
        'F1-score (micro)': cv_results['test_f1_micro'],
        'Precision': cv_results['test_precision_micro'],
        'Recall': cv_results['test_recall_micro']
    })

    stats = pd.DataFrame([
        {
            'Фолд': 'СРЕДНЕЕ (Mean)',
            'F1-score (micro)': metrics_df['F1-score (micro)'].mean(),
            'Precision': metrics_df['Precision'].mean(),
            'Recall': metrics_df['Recall'].mean()
        },
        {
            'Фолд': 'ОТКЛОНЕНИЕ (Std)',
            'F1-score (micro)': metrics_df['F1-score (micro)'].std(),
            'Precision': metrics_df['Precision'].std(),
            'Recall': metrics_df['Recall'].std()
        }
    ])

    full_report = pd.concat([metrics_df, stats], ignore_index=True)

    print("Метрики кросс-валидации:")
    print(full_report.round(4))


    return pipeline, full_report



In [ ]:
model_multioutput =  predict_ext_with_cv(df, 'logreg')

5-fold CV для MultiOutput (LOGREG)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.65       0.59    0.74
1            Fold 2              0.67       0.59    0.76
2            Fold 3              0.65       0.58    0.73
3            Fold 4              0.66       0.59    0.75
4            Fold 5              0.64       0.57    0.73
5    СРЕДНЕЕ (Mean)              0.65       0.58    0.74
6  ОТКЛОНЕНИЕ (Std)              0.01       0.01    0.01


In [ ]:
model_multioutput =  predict_ext_with_cv(df, 'xgb')

5-fold CV для MultiOutput (XGB)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.67       0.80    0.57
1            Fold 2              0.64       0.74    0.56
2            Fold 3              0.66       0.77    0.58
3            Fold 4              0.66       0.80    0.56
4            Fold 5              0.63       0.78    0.53
5    СРЕДНЕЕ (Mean)              0.65       0.78    0.56
6  ОТКЛОНЕНИЕ (Std)              0.02       0.02    0.02


In [ ]:
model_multioutput =  predict_ext_with_cv(df, 'rf')

5-fold CV для MultiOutput (RF)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.63       0.88    0.50
1            Fold 2              0.66       0.87    0.53
2            Fold 3              0.63       0.85    0.50
3            Fold 4              0.61       0.84    0.48
4            Fold 5              0.60       0.84    0.46
5    СРЕДНЕЕ (Mean)              0.63       0.85    0.49
6  ОТКЛОНЕНИЕ (Std)              0.02       0.02    0.03


In [ ]:
model_multioutput =  predict_ext_with_cv(df, 'svc')

5-fold CV для MultiOutput (SVC)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.69       0.73    0.66
1            Fold 2              0.69       0.69    0.69
2            Fold 3              0.69       0.71    0.67
3            Fold 4              0.69       0.72    0.66
4            Fold 5              0.68       0.72    0.65
5    СРЕДНЕЕ (Mean)              0.69       0.71    0.67
6  ОТКЛОНЕНИЕ (Std)              0.00       0.01    0.01


In [ ]:
model_multioutput =  predict_ext_with_cv(df_input, 'svc')
model_multioutput =  predict_ext_with_cv(df_input, 'rf')
model_multioutput =  predict_ext_with_cv(df_input, 'xgb')
model_multioutput =  predict_ext_with_cv(df_input, 'logreg')

5-fold CV для MultiOutput (SVC)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.64       0.65    0.63
1            Fold 2              0.64       0.62    0.66
2            Fold 3              0.66       0.65    0.66
3            Fold 4              0.64       0.64    0.65
4            Fold 5              0.64       0.65    0.63
5    СРЕДНЕЕ (Mean)              0.64       0.64    0.65
6  ОТКЛОНЕНИЕ (Std)              0.01       0.01    0.02
5-fold CV для MultiOutput (RF)...
Метрики кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.60       0.76    0.49
1            Fold 2              0.61       0.77    0.50
2            Fold 3              0.59       0.74    0.48
3            Fold 4              0.59       0.76    0.48
4            Fold 5              0.56       0.72    0.45
5    СРЕДНЕЕ (Mean)              0.59       0.75    0.48
6  ОТКЛОНЕНИЕ (Std)       

## Classifier Chain

In [ ]:
def predict_chain(df_given, classifier_type, order='random'):
    X = df_given[['comment', 'Длина комментария']]
    y = df_given.drop(['comment', 'Длина комментария', 'id'], axis=1).fillna(0).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), 'comment'),
            ('num', StandardScaler(), ['Длина комментария'])
        ]
    )


    if classifier_type == 'logreg':
        clf = LogisticRegression(solver='liblinear', class_weight='balanced')
    elif classifier_type == 'svc':
        clf = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)
    elif classifier_type == 'rf':
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1)
    elif classifier_type == 'xgb':
        clf = XGBClassifier(eval_metric='logloss', use_label_encoder=False)
    else:
        raise ValueError("Unknown classifier type")


    base_lr = clf
    chain = ClassifierChain(base_lr, order=order, random_state=42)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', chain)
    ])


    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"Результаты ClassifierChain", base_lr)
    print(classification_report(y_test, y_pred, target_names=y.columns, zero_division=0))

    return pipeline


model_chain = predict_chain(df, 'logreg')

In [ ]:
model_chain = predict_chain(df, 'svc')

In [ ]:
model_chain = predict_chain(df, 'xgb')

### 5-fold

In [ ]:

def train_and_validate_chain(df_given, classifier_type='logreg', order='random'):
    """
    Функция для обучения ClassifierChain с 5-кратной кросс-валидацией.
    Выводит метрики по фолдам и сохраняет обученную модель.
    """
    drop_cols = ['comment', 'Длина комментария', 'id']

    X = df_given[['comment', 'Длина комментария']]
    y = df_given.drop(columns=[col for col in drop_cols if col in df_given.columns]).fillna(0).astype(int)

    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), 'comment'),
            ('num', StandardScaler(), ['Длина комментария'])
        ]
    )

    if classifier_type == 'logreg':
        base_clf = LogisticRegression(solver='liblinear', class_weight='balanced')
    elif classifier_type == 'svc':
        base_clf = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)
    elif classifier_type == 'rf':
        base_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1)
    elif classifier_type == 'xgb':
        base_clf = XGBClassifier(eval_metric='logloss')
    else:
        raise ValueError(f"Неизвестный тип классификатора: {classifier_type}")

    chain = ClassifierChain(base_clf, order=order, random_state=42)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', chain)
    ])

    # кросс валидация, 5 фолдов
    print(f"5-fold CV для {classifier_type}")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    scoring = ['f1_micro', 'precision_micro', 'recall_micro']
    cv_results = cross_validate(pipeline, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    metrics_df = pd.DataFrame({
        'Фолд': [f'Fold {i+1}' for i in range(5)],
        'F1-score (micro)': cv_results['test_f1_micro'],
        'Precision': cv_results['test_precision_micro'],
        'Recall': cv_results['test_recall_micro']
    })

    stats = pd.DataFrame([
        {
            'Фолд': 'СРЕДНЕЕ (Mean)',
            'F1-score (micro)': metrics_df['F1-score (micro)'].mean(),
            'Precision': metrics_df['Precision'].mean(),
            'Recall': metrics_df['Recall'].mean()
        },
        {
            'Фолд': 'ОТКЛОНЕНИЕ (Std)',
            'F1-score (micro)': metrics_df['F1-score (micro)'].std(),
            'Precision': metrics_df['Precision'].std(),
            'Recall': metrics_df['Recall'].std()
        }
    ])

    final_report = pd.concat([metrics_df, stats], ignore_index=True)
    print("\n Статистика кросс-валидации:")
    print(final_report.round(4))

    return pipeline, final_report


In [ ]:
model_chain = train_and_validate_chain(df, 'xgb')

5-fold CV для xgb

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.65       0.76    0.56
1            Fold 2              0.68       0.75    0.61
2            Fold 3              0.64       0.74    0.57
3            Fold 4              0.65       0.79    0.55
4            Fold 5              0.65       0.77    0.56
5    СРЕДНЕЕ (Mean)              0.65       0.76    0.57
6  ОТКЛОНЕНИЕ (Std)              0.01       0.02    0.02


In [ ]:
model_chain = train_and_validate_chain(df, 'svc')

5-fold CV для svc

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.69       0.71    0.67
1            Fold 2              0.69       0.70    0.69
2            Fold 3              0.68       0.69    0.67
3            Fold 4              0.67       0.68    0.66
4            Fold 5              0.68       0.70    0.65
5    СРЕДНЕЕ (Mean)              0.68       0.70    0.67
6  ОТКЛОНЕНИЕ (Std)              0.01       0.01    0.01


In [ ]:
model_chain = train_and_validate_chain(df, 'rf')

5-fold CV для rf

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.64       0.85    0.51
1            Fold 2              0.68       0.86    0.55
2            Fold 3              0.62       0.83    0.50
3            Fold 4              0.63       0.85    0.50
4            Fold 5              0.61       0.85    0.48
5    СРЕДНЕЕ (Mean)              0.64       0.85    0.51
6  ОТКЛОНЕНИЕ (Std)              0.02       0.01    0.03


In [ ]:
model_chain = train_and_validate_chain(df, 'logreg')

5-fold CV для logreg

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.62       0.56    0.71
1            Fold 2              0.62       0.56    0.69
2            Fold 3              0.62       0.56    0.70
3            Fold 4              0.61       0.55    0.70
4            Fold 5              0.62       0.56    0.71
5    СРЕДНЕЕ (Mean)              0.62       0.56    0.70
6  ОТКЛОНЕНИЕ (Std)              0.01       0.00    0.01


In [ ]:
df_input.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2583 entries, 0 to 2582
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   id                                   2583 non-null   int64 
 1   Длина комментария                    2583 non-null   int64 
 2   Зарплата                             2583 non-null   int64 
 3   Дефицит                              2583 non-null   int64 
 4   Коллеги                              2583 non-null   int64 
 5   Востребованность работы              2583 non-null   int64 
 6   Содержание работы                    2583 non-null   int64 
 7   Руководство и репутация организации  2583 non-null   int64 
 8   Инфраструктура региона               2583 non-null   int64 
 9   Возможности для развития             2583 non-null   int64 
 10  Профессиональные риски               2583 non-null   int64 
 11  Стабильность и защищенность          2583 n

In [ ]:
model_multioutput =  train_and_validate_chain(df_input, 'svc')
model_multioutput =  train_and_validate_chain(df_input, 'rf')
model_multioutput =  train_and_validate_chain(df_input, 'xgb')
model_multioutput =  train_and_validate_chain(df_input, 'logreg')

5-fold CV для svc

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.63       0.66    0.60
1            Fold 2              0.65       0.67    0.63
2            Fold 3              0.67       0.69    0.64
3            Fold 4              0.66       0.69    0.63
4            Fold 5              0.63       0.67    0.60
5    СРЕДНЕЕ (Mean)              0.65       0.68    0.62
6  ОТКЛОНЕНИЕ (Std)              0.02       0.01    0.02
5-fold CV для rf

 Статистика кросс-валидации:
               Фолд  F1-score (micro)  Precision  Recall
0            Fold 1              0.61       0.81    0.49
1            Fold 2              0.64       0.82    0.52
2            Fold 3              0.61       0.81    0.49
3            Fold 4              0.62       0.84    0.49
4            Fold 5              0.56       0.78    0.44
5    СРЕДНЕЕ (Mean)              0.61       0.81    0.49
6  ОТКЛОНЕНИЕ (Std)              0.03       0.02  

# Сохранение параметров модели

In [ ]:

X = df[['comment', 'Длина комментария']]
y = df.drop(['comment', 'Длина комментария', 'id'], axis=1).fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=10000, ngram_range=(1, 2)), 'comment'),
        ('num', StandardScaler(), ['Длина комментария'])
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', MultiOutputClassifier(clf))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred, target_names=y.columns, zero_division=0))



In [ ]:
y.columns

Index(['Зарплата', 'Дефицит', 'Возможности для развития', 'Рабочие процессы',
       'Условия работы', 'График работы', 'Социальный пакет',
       'Нет комментариев', 'Комментарии к опроснику',
       'Благодарность за опросник', 'Скептицизм к изменениям',
       'Надежда на изменения', 'Позитив-неуточненно',
       'Руководство, коллеги и репутация организации',
       'Взаимодействие с пациентами, проф риски, востребованность',
       'Инфраструктура и здравоохранение региона'],
      dtype='object')

In [ ]:
import joblib

joblib.dump(pipeline, 'LinearSVC_classifier_model_11may.pkl')


['LinearSVC_classifier_model_11may.pkl']